In [ ]:
!pip install -q \
  "numpy==1.26.4" \
  "scipy==1.13.1" \
  "scikit-learn==1.4.2" \
  "opencv-python-headless==4.9.0.80" \
  "ultralytics>=8.3.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.2/38.2 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incomp

In [ ]:
import numpy as np
print("numpy", np.__version__, np.__file__)

import scipy
print("scipy", scipy.__version__)

from ultralytics import YOLO
print("ultralytics OK")

numpy 1.26.4 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
scipy 1.13.1
ultralytics OK


In [ ]:
!git clone https://github.com/Aaryan0504/Fault_Detection /content/ups_yolo

Cloning into '/content/ups_yolo'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 59 (delta 26), reused 55 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 49.03 KiB | 880.00 KiB/s, done.
Resolving deltas: 100% (26/26), done.


In [ ]:
from pathlib import Path
import os
import shutil

PROJECT_ROOT = Path("/content/ups_yolo")  # or your clone path
os.chdir(PROJECT_ROOT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")
print("MyDrive exists:", root.is_dir())

# List top-level entries (first 50)
for p in sorted(root.iterdir())[:50]:
    print(p.name, "->", "dir" if p.is_dir() else "file")

MyDrive exists: True
Colab Notebooks -> dir
data -> dir
labels -> dir
train -> dir


In [ ]:
from pathlib import Path

start = Path("/content/drive/MyDrive")
for p in start.rglob("*"):
    if not p.is_dir():
        continue
    if p.name == "data/augmented":
        print(p)


In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/ups_yolo")

# Source folders in Google Drive
DRIVE_RAW = Path("/content/drive/MyDrive/data/raw")
DRIVE_AUG = Path("/content/drive/MyDrive/data/augmented")


def copy_data_dir(name: str, src: Path) -> None:
    dst = PROJECT_ROOT / "data" / name

    if not src.is_dir():
        raise FileNotFoundError(f"Not a folder: {src}")

    # ✅ Check YOLO structure
    if not (src / "images" / "train").exists():
        raise ValueError(f"{name}: missing images/train")

    if not (src / "labels" / "train").exists():
        raise ValueError(f"{name}: missing labels/train")

    dst.parent.mkdir(parents=True, exist_ok=True)

    # Remove old folder if exists
    if dst.exists():
        shutil.rmtree(dst)

    shutil.copytree(src, dst)

    print(f"✅ Copied {name}: {src} -> {dst}")

print("DRIVE_RAW =", DRIVE_RAW)

import os
for root, dirs, files in os.walk(DRIVE_RAW):
    print(root)
    break

# Copy RAW dataset
copy_data_dir("raw", DRIVE_RAW)

# Copy AUGMENTED dataset
copy_data_dir("augmented", DRIVE_AUG)

DRIVE_RAW = /content/drive/MyDrive/data/raw
/content/drive/MyDrive/data/raw
✅ Copied raw: /content/drive/MyDrive/data/raw -> /content/ups_yolo/data/raw
✅ Copied augmented: /content/drive/MyDrive/data/augmented -> /content/ups_yolo/data/augmented


In [ ]:
assert (PROJECT_ROOT / "data" / "augmented" / "images" / "train").is_dir()

assert (PROJECT_ROOT / "configs" / "yolo26s_finetune.yaml").is_file()

In [ ]:
import yaml

ds_path = PROJECT_ROOT / "dataset.yaml"
cfg = yaml.safe_load(ds_path.read_text(encoding="utf-8"))

# ✅ Ensure required keys exist
required_keys = ["train", "val"]
for k in required_keys:
    if k not in cfg:
        raise ValueError(f"Missing '{k}' in dataset.yaml")

# ✅ Point to AUGMENTED dataset
cfg["path"] = str((PROJECT_ROOT / "data" / "augmented").resolve())

ds_path.write_text(
    yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8"
)

ds = str(ds_path.resolve())
yolo_cfg = str(PROJECT_ROOT / "configs" / "yolo26s_finetune.yaml")

print("data:", cfg["path"])

data: /content/ups_yolo/data/augmented


In [ ]:
import torch
train_device = 0 if torch.cuda.is_available() else "cpu"
print(train_device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

0 Tesla T4


In [ ]:
from pathlib import Path

img_dir = PROJECT_ROOT / "data" / "augmented" / "images"
lbl_dir = PROJECT_ROOT / "data" / "augmented" / "labels"

for split in ["train", "val", "test"]:
    imgs = (img_dir / split).glob("*.*")

    for img in imgs:
        lbl = lbl_dir / split / (img.stem + ".txt")

        if not lbl.exists():
            lbl.parent.mkdir(parents=True, exist_ok=True)
            lbl.touch()

print("✅ Empty labels created for no-fault images")

✅ Empty labels created for no-fault images


In [ ]:
from pathlib import Path

labels_root = PROJECT_ROOT / "data" / "augmented" / "labels"

# ---------------------------------------------------
# Filename keyword -> correct class ID
# ---------------------------------------------------

rules = {
    "Input_Cable_Fault": 0,
    "Loose_Connection": 1,
    "Output_Cable_Fault": 2,
    "RI_Cable_Mismatch": 3,
    "Screw_Fault": 4,
    "Signal_Cable_Fault": 5,
    "Cable_Mismatch": 6,
    "Red_White_Mismatch": 7,
    "Ferrule_Mismatch": 8,
}

# ---------------------------------------------------
# Process train / val / test
# ---------------------------------------------------

for split in ["train", "val", "test"]:

    split_dir = labels_root / split

    txt_files = list(split_dir.glob("*.txt"))

    for txt_file in txt_files:

        filename = txt_file.stem.lower()

        target_class = None

        # Match filename with class keyword
        for keyword, cls_id in rules.items():

            if keyword.lower() in filename:
                target_class = cls_id
                break

        # Skip unrelated files
        if target_class is None:
            continue

        with open(txt_file, "r") as f:
            lines = f.readlines()

        updated_lines = []

        for line in lines:

            parts = line.strip().split()

            # Skip empty or invalid lines
            if len(parts) < 5:
                continue

            # Replace class ID
            parts[0] = str(target_class)

            updated_lines.append(" ".join(parts))

        # Write updated labels
        with open(txt_file, "w") as f:
            f.write("\n".join(updated_lines))

        print(f"✅ {txt_file.name} -> class {target_class}")

print("🎉 All labels updated successfully.")

✅ Signal_Cable_Fault_8_Effects_aug02.txt -> class 5
✅ Cable_Mismatch_127.txt -> class 6
✅ Red_White_Mismatch_36.txt -> class 7
✅ Input_Cable_Fault_1_aug03.txt -> class 0
✅ Loose_Connection_1_aug04.txt -> class 1
✅ Loose_Connection_1_aug01.txt -> class 1
✅ Input_Cable_Fault_4.txt -> class 0
✅ RI_Cable_mismatch_27_aug00.txt -> class 3
✅ Cable_Mismatch_40.txt -> class 6
✅ Screw_Fault_5_Zoomed_aug03.txt -> class 4
✅ Input_Cable_Fault_7.txt -> class 0
✅ Cable_Mismatch_26.txt -> class 6
✅ Signal_Cable_Fault_8_Effects_2_aug00.txt -> class 5
✅ Red_White_Mismatch_131.txt -> class 7
✅ Input_Cable_Fault_2_aug00.txt -> class 0
✅ Loose_Connection_9_Zoomed.txt -> class 1
✅ Cable_Mismatch_128.txt -> class 6
✅ Screw_Fault_3_aug00.txt -> class 4
✅ Red_White_Mismatch_139.txt -> class 7
✅ Input_Cable_Fault_14_aug02.txt -> class 0
✅ Loose_Connection_2_aug03.txt -> class 1
✅ Red_White_Mismatch_120.txt -> class 7
✅ Red_White_Mismatch_28.txt -> class 7
✅ Screw_Fault_16_aug01.txt -> class 4
✅ Input_Cable_Faul

In [ ]:
!python scripts/train_phase_a.py

INFO:__main__:Train kwargs: {'data': 'dataset.yaml', 'epochs': 20, 'imgsz': 640, 'batch': 16, 'freeze': 10, 'task': 'detect', 'cfg': 'configs/yolo26s_finetune.yaml', 'project': 'runs', 'name': 'phase_a', 'exist_ok': True, 'val': True, 'plots': True, 'save': True, 'device': '0', 'save_dir': '/content/ups_yolo/runs/phase_a'}
                           Phase A — Training summary                           
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field         ┃ Value                                                        ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Model variant │ yolo11m (COCO detect pretrained)                             │
│ Task          │ detect (axis-aligned bbox)                                   │
│ Classes       │ 6 — input_cable_fault, loose_connection, output_cable_fault, │
│               │ ri_cable_mismatch, screw_faults, signal_cable_mismatch,      │
│               │ J14_cable

In [ ]:
!python scripts/train_phase_b.py

INFO:__main__:Train kwargs: {'data': 'dataset.yaml', 'epochs': 100, 'imgsz': 640, 'batch': 8, 'freeze': 0, 'task': 'detect', 'cfg': 'configs/yolo26s_finetune.yaml', 'lr0': 0.0001, 'lrf': 0.01, 'cos_lr': True, 'project': 'runs', 'name': 'phase_b', 'exist_ok': True, 'val': True, 'plots': True, 'save': True, 'patience': 15, 'device': '0', 'save_dir': '/content/ups_yolo/runs/phase_b'}
                         Phase B — Training summary                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Field                   ┃ Value                                          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Loaded from             │ /content/ups_yolo/runs/phase_a/weights/best.pt │
│ Task                    │ detect                                         │
│ Epochs                  │ 100                                            │
│ Batch size              │ 8                                              │


In [ ]:
%cd /content/ups_yolo
!python -c "from pathlib import Path; p=Path('scripts/validate_model.py'); s=p.read_text(encoding='utf-8'); s=s.replace('tmp = out_path.with_suffix(out_path.suffix + \".tmp\")','tmp = out_path.with_name(out_path.stem + \".tmp\" + out_path.suffix)'); p.write_text(s, encoding='utf-8'); print('patched ok')"

/content
patched ok


In [ ]:
!python -c "from pathlib import Path; lines=Path('scripts/validate_model.py').read_text(encoding='utf-8').splitlines(); print('\\n'.join(lines[133:139]))"

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    # Matplotlib infers output format from the file extension, so ensure the
    # temporary file still ends with the target suffix (e.g. ".png").
    tmp = out_path.with_name(out_path.stem + ".tmp" + out_path.suffix)
    fig.savefig(tmp, dpi=200)


In [ ]:
%cd /content/ups_yolo
!yolo task=detect mode=val model="runs/phase_b/weights/best.pt" data="dataset.yaml" split=test imgsz=640 plots=False

/content
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,036,971 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 231.3±26.6 MB/s, size: 3612.1 KB)
val: Scanning /content/ups_yolo/data/augmented/labels/test... 42 images, 13 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 42/42 8.0it/s 5.3s
val: /content/ups_yolo/data/augmented/images/test/Cable_Mismatch_147.jpg: corrupt JPEG restored and saved
val: /content/ups_yolo/data/augmented/images/test/Cable_Mismatch_148.jpg: corrupt JPEG restored and saved
val: /content/ups_yolo/data/augmented/images/test/Cable_Mismatch_149.jpg: corrupt JPEG restored and saved
val: /content/ups_yolo/data/augmented/images/test/Cable_Mismatch_150.jpg: corrupt JPEG restored and saved
val: /content/ups_yolo/data/augmented/images/test/Ferrule_Mismatch_146.jpg: corrupt JPEG restored and saved
val: /content/ups_yolo/data/augmented/images/test/Ferrule_Mismatch_147

In [ ]:
!yolo task=detect mode=predict model="runs/phase_b/weights/best.pt" source="/content/ups_yolo/data/augmented/images/val/Output_Cable_Fault_13.png" imgsz=640 conf=0.01 iou=0.7 save=True

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,036,971 parameters, 0 gradients, 67.7 GFLOPs

image 1/1 /content/ups_yolo/data/augmented/images/val/Output_Cable_Fault_13.png: 640x480 5 output_cable_faults, 1 signal_cable_mismatch, 98.8ms
Speed: 6.9ms preprocess, 98.8ms inference, 18.9ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/ups_yolo/runs/detect/predict
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [ ]:
from pathlib import Path
from google.colab import files

PROJECT_ROOT = Path("/content/ups_yolo")  # or /content/drive/MyDrive/ups_yolo

zip_path = Path("/content/ups_yolo_training_export.zip")
!cd {PROJECT_ROOT} && zip -r {zip_path} runs/phase_a runs/phase_b
files.download(str(zip_path))

updating: runs/phase_a/ (stored 0%)
updating: runs/phase_a/labels.jpg (deflated 31%)
updating: runs/phase_a/confusion_matrix.png (deflated 20%)
updating: runs/phase_a/train_batch1.jpg (deflated 2%)
updating: runs/phase_a/train_batch2.jpg (deflated 2%)
updating: runs/phase_a/train_batch690.jpg (deflated 3%)
updating: runs/phase_a/confusion_matrix_normalized.png (deflated 19%)
updating: runs/phase_a/BoxP_curve.png (deflated 9%)
updating: runs/phase_a/BoxPR_curve.png (deflated 16%)
updating: runs/phase_a/BoxF1_curve.png (deflated 7%)
updating: runs/phase_a/train_batch692.jpg (deflated 6%)
updating: runs/phase_a/val_batch0_pred.jpg (deflated 10%)
updating: runs/phase_a/weights/ (stored 0%)
updating: runs/phase_a/weights/epoch10.pt (deflated 8%)
updating: runs/phase_a/weights/best.pt (deflated 8%)
updating: runs/phase_a/weights/epoch0.pt (deflated 8%)
updating: runs/phase_a/weights/last.pt (deflated 8%)
updating: runs/phase_a/results.csv (deflated 68%)
updating: runs/phase_a/train_batch0.jp

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>